# Creative Tasks — Multi-Field Signatures & Generation

Optimize a prompt that generates movie taglines from plot summaries, keywords, and genres.

**Dataset:** TMDB Movies — 150 films with overview, keywords, genres, and tagline.

**What you'll learn:**
1. Define multi-input signatures (3 input fields → 1 output)
2. Write an LLM-as-judge metric for open-ended generation
3. Validate your code before submitting (the `/validate-code` endpoint)
4. Use the serve API for inference without downloading the artifact

**Prerequisites:** Complete [01_quickstart.ipynb](01_quickstart.ipynb) first.

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set the OPENAI_API_KEY environment variable.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": "https://api.openai.com/v1",
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": OPENAI_API_KEY},
}

dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_key=OPENAI_API_KEY,
    )
)

client = DSPyServiceClient(BASE_URL)
client.health()

In [ ]:
# Browse available models in the catalog
models = client.models()
for m in models[:5]:
    print(f"  {m['name']} — {m.get('provider', '')}")

## 1. Load Dataset

Each row has 3 inputs (overview, keywords, genres) and 1 output (tagline).

In [ ]:
with open(Path("../../data/tmdb_movies_150.json")) as f:
    dataset = json.load(f)

print(f"{len(dataset)} movies")
print(f"Fields: {list(dataset[0].keys())}")
print()
sample = dataset[0]
print(f"Overview:  {sample['overview'][:100]}...")
print(f"Keywords:  {sample['keywords']}")
print(f"Genres:    {sample['genres']}")
print(f"Tagline:   {sample['tagline']}")

## 2. Multi-Field Signature

Unlike the quickstart's single-input signature, this one maps three input fields to one output.

In [ ]:
class TaglineGenerator(dspy.Signature):
    """Generate a catchy, memorable movie tagline from the plot summary, keywords, and genres."""
    overview: str = dspy.InputField(desc="Brief plot summary of the movie")
    keywords: str = dspy.InputField(desc="Comma-separated thematic keywords")
    genres: str = dspy.InputField(desc="Comma-separated genre labels")
    tagline: str = dspy.OutputField(desc="A short, catchy movie tagline")


SIGNATURE_CODE = serialize_source(TaglineGenerator)
print(SIGNATURE_CODE)

## 3. LLM-as-Judge Metric

For creative tasks there's no exact-match answer. Instead, we use the LLM itself to judge quality on a 0-1 scale.

The metric asks: is the generated tagline relevant, concise, and catchy compared to the reference?

In [ ]:
METRIC_CODE = '''
def tagline_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    reference = (gold.tagline or "").strip()
    generated = (pred.tagline or "").strip()

    if not generated:
        return dspy.Prediction(score=0.0, feedback="Empty tagline.")

    # Exact or near match gets full score
    if reference.lower() == generated.lower():
        return dspy.Prediction(score=1.0, feedback="Exact match with reference tagline.")

    # Use a lightweight heuristic: reward brevity and keyword overlap
    ref_words = set(reference.lower().split())
    gen_words = set(generated.lower().split())
    overlap = len(ref_words & gen_words) / max(len(ref_words), 1)

    # Penalize overly long taglines (good taglines are under 15 words)
    length_penalty = max(0, 1.0 - max(0, len(generated.split()) - 15) * 0.1)

    score = round(min(1.0, overlap * 0.7 + length_penalty * 0.3), 2)

    feedback = f"Reference: '{reference}'. Generated: '{generated}'. "
    feedback += f"Word overlap: {overlap:.0%}, length: {len(generated.split())} words."
    if score < 0.5:
        feedback += " Try capturing the core theme in fewer, punchier words."

    return dspy.Prediction(score=score, feedback=feedback)
'''

## 4. Validate Before Submitting

The `/validate-code` endpoint checks your signature and metric code for syntax errors and field mismatches without starting a job.

In [ ]:
validation = client.validate_code({
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "column_mapping": {
        "inputs": {"overview": "overview", "keywords": "keywords", "genres": "genres"},
        "outputs": {"tagline": "tagline"},
    },
})

print(f"Valid: {validation['valid']}")
if validation.get("errors"):
    print(f"Errors: {validation['errors']}")
if validation.get("warnings"):
    print(f"Warnings: {validation['warnings']}")
if validation.get("signature_fields"):
    print(f"Fields: {validation['signature_fields']}")

## 5. Submit Optimization

Note the column mapping: each signature field maps to the corresponding JSON key in the dataset.

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "light", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"overview": "overview", "keywords": "keywords", "genres": "genres"},
        "outputs": {"tagline": "tagline"},
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "model_config": MODEL_CONFIG,
    "reflection_model_config": MODEL_CONFIG,
}

optimization_id = client.submit(payload)
print(f"Submitted: {optimization_id}")

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=5)

if result["status"] == "success":
    r = result["result"]
    print(f"\nBaseline score:  {r.get('baseline_test_metric', 'N/A')}")
    print(f"Optimized score: {r.get('optimized_test_metric', 'N/A')}")
    print(f"Runtime: {r.get('runtime_seconds', 0):.0f}s")
else:
    print(f"\nFailed: {result.get('message')}")

## 6. Serve the Optimized Program (Remote Inference)

Instead of downloading the artifact and running locally, you can call the `/serve` endpoint to run inference on the server. This is useful when you don't have `dspy` installed or want to share the endpoint.

In [ ]:
# Check serve availability
serve_info = client.serve_info(optimization_id)
serve_info

In [ ]:
test_movies = [
    {
        "overview": "A young wizard discovers he is the chosen one destined to defeat an ancient dark lord.",
        "keywords": "wizard, prophecy, dark lord, magic school, friendship",
        "genres": "Fantasy, Adventure",
    },
    {
        "overview": "A retired detective is pulled back into the game when a serial killer begins leaving cryptic puzzles.",
        "keywords": "detective, serial killer, puzzles, retirement, obsession",
        "genres": "Thriller, Mystery",
    },
    {
        "overview": "Two rival chefs compete for a prestigious culinary award while discovering they have more in common than they thought.",
        "keywords": "cooking, rivalry, romance, competition, food",
        "genres": "Comedy, Romance",
    },
]

for movie in test_movies:
    response = client.serve(optimization_id, movie)
    print(f"Genres:  {movie['genres']}")
    print(f"Tagline: {response.get('tagline', response)}")
    print()

## 7. Local Inference (Alternative)

You can also download the program and run it locally with your own `dspy.LM`.

In [ ]:
program = client.load_program(optimization_id)

response = program(
    overview="An astronaut stranded on Mars must use science to survive until rescue arrives.",
    keywords="mars, survival, science, isolation, nasa",
    genres="Sci-Fi, Drama",
)
print(f"Tagline: {response.tagline}")

In [ ]:
dspy.inspect_history(n=1)

## What's Next

- **API Reference** — `http://localhost:8000/reference` for all endpoints including templates, analytics, and bulk operations.
- **Frontend Dashboard** — `http://localhost:3001` to view, compare, and manage optimizations visually.